# CheXpert Frozen-Protocol -- Results Dashboard (read-only)

Purely displays whatever training results already exist on Drive, across all 6 architectures.
**No training, no GPU needed, safe to run any time.** Reads each experiment's `results.txt`
directly (written by `engine.py` once that tier's full seed-set has finished testing) -- so
an experiment only shows up here once it's actually complete, never mid-training.

Covers: Ark+ (+ its class-weighted BCE follow-up), SwinL, SwinB, DenseNet-121, RadDino,
ConvNeXtV2 -- screening and headline tiers each. Drive paths and experiment names below are
copied exactly from each model's own training notebook, not guessed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, re, ast
import pandas as pd

# ===== Registry: exactly matches each model's own training notebook (DRIVE_CKPT_DIR, exp_name) =====
# Ark+ and SwinL share one Drive folder since SwinL is Ark+'s architecture-matched control,
# set up together in the same notebook family.
_ARKPLUS_DIR = '/content/drive/MyDrive/ark_plus_chexpert_checkpoints'
EXPERIMENTS = [
    {"model": "Ark+",                  "tier": "screening", "dir": _ARKPLUS_DIR, "exp_name": "_protocol1_arkplus_screening"},
    {"model": "Ark+",                  "tier": "headline",  "dir": _ARKPLUS_DIR, "exp_name": "_protocol1_arkplus_headline"},
    {"model": "Ark+ (class-weighted)", "tier": "screening", "dir": _ARKPLUS_DIR, "exp_name": "_protocol1_arkplus_cw_screening"},
    {"model": "SwinL (control)",       "tier": "screening", "dir": _ARKPLUS_DIR, "exp_name": "_protocol1_swinl_screening"},
    {"model": "SwinL (control)",       "tier": "headline",  "dir": _ARKPLUS_DIR, "exp_name": "_protocol1_swinl_headline"},
    {"model": "SwinB (V2)",            "tier": "screening", "dir": "/content/drive/MyDrive/swinb_chexpert_checkpoints", "exp_name": "_protocol1_swinv2b_screening"},
    {"model": "SwinB (V2)",            "tier": "headline",  "dir": "/content/drive/MyDrive/swinb_chexpert_checkpoints", "exp_name": "_protocol1_swinv2b_headline"},
    {"model": "DenseNet-121",          "tier": "screening", "dir": "/content/drive/MyDrive/densenet121_chexpert_checkpoints", "exp_name": "_protocol1_densenet121_screening"},
    {"model": "DenseNet-121",          "tier": "headline",  "dir": "/content/drive/MyDrive/densenet121_chexpert_checkpoints", "exp_name": "_protocol1_densenet121_headline"},
    {"model": "RadDino",               "tier": "screening", "dir": "/content/drive/MyDrive/raddino_chexpert_checkpoints", "exp_name": "_protocol1_raddino_screening"},
    {"model": "RadDino",               "tier": "headline",  "dir": "/content/drive/MyDrive/raddino_chexpert_checkpoints", "exp_name": "_protocol1_raddino_headline"},
    {"model": "ConvNeXtV2",            "tier": "screening", "dir": "/content/drive/MyDrive/convnextv2_chexpert_checkpoints", "exp_name": "_protocol1_convnextv2_screening"},
    {"model": "ConvNeXtV2",            "tier": "headline",  "dir": "/content/drive/MyDrive/convnextv2_chexpert_checkpoints", "exp_name": "_protocol1_convnextv2_headline"},
]

COMPETITION_5 = ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Pleural Effusion']

_BLOCK_RE = re.compile(
    r"AUC\s*=\s*(?P<auc>\[[^\]]*\])\s*"
    r"MCC\s*=\s*(?P<mcc>\[[^\]]*\])\s*"
    r"AP\s*=\s*(?P<ap>\[[^\]]*\])\s*"
    r"F1\s*=\s*(?P<f1>\[[^\]]*\])\s*"
    r"Recall\s*=\s*(?P<recall>\[[^\]]*\])\s*"
    r"Accuracy\s*=\s*(?P<accuracy>\[[^\]]*\])\s*"
    r"ECE\s*=\s*(?P<ece>\[[^\]]*\])"
)

def _parse_array(text):
    # results.txt sometimes reprs numpy scalars as np.float32(0.123) instead of plain 0.123 --
    # strip that wrapper before literal_eval.
    cleaned = re.sub(r"np\.float(32|64)\(([^)]+)\)", r"\2", text)
    return [float(x) for x in ast.literal_eval(cleaned)]

def parse_results_txt(path):
    """One dict per completed run (seed) found in this experiment's results.txt."""
    if not os.path.exists(path):
        return []
    text = open(path, encoding="utf-8").read()
    runs = []
    for m in _BLOCK_RE.finditer(text):
        runs.append({k: _parse_array(m.group(k)) for k in ["auc", "mcc", "ap", "f1", "recall", "accuracy", "ece"]})
    return runs

In [ ]:
rows = []
per_class_rows = []
missing = []

for exp in EXPERIMENTS:
    results_path = os.path.join(exp["dir"], "Outputs/Classification/CheXpert", exp["exp_name"] + "_results.txt")
    runs = parse_results_txt(results_path)
    if not runs:
        missing.append(f"{exp['model']} -- {exp['tier']}")
        continue
    for seed_idx, run in enumerate(runs):
        rows.append({
            "model": exp["model"], "tier": exp["tier"], "run": seed_idx,
            "AUC": sum(run["auc"]) / len(run["auc"]),
            "F1": sum(run["f1"]) / len(run["f1"]),
            "Sensitivity": sum(run["recall"]) / len(run["recall"]),
            "Accuracy": sum(run["accuracy"]) / len(run["accuracy"]),
            "MCC": sum(run["mcc"]) / len(run["mcc"]),
            "AP": sum(run["ap"]) / len(run["ap"]),
            "ECE": sum(run["ece"]) / len(run["ece"]),
        })
        for i, cls in enumerate(COMPETITION_5):
            per_class_rows.append({
                "model": exp["model"], "tier": exp["tier"], "run": seed_idx, "class": cls,
                "AUC": run["auc"][i], "F1": run["f1"][i], "Sensitivity": run["recall"][i],
                "Accuracy": run["accuracy"][i], "MCC": run["mcc"][i], "AP": run["ap"][i], "ECE": run["ece"][i],
            })

summary_df = pd.DataFrame(rows)
per_class_df = pd.DataFrame(per_class_rows)
_found = summary_df[["model", "tier"]].drop_duplicates().shape[0] if len(rows) else 0
print(f"Found completed results for {_found} of {len(EXPERIMENTS)} registered experiments.")

In [ ]:
print("Per-seed results -- not pooled into a single number: a mean+/-range across only 1-2 seeds")
print("would misrepresent the spread, matching this project's own protocol convention.")
display(summary_df.sort_values(["model", "tier", "run"]).reset_index(drop=True) if len(rows) else "Nothing completed yet.")

In [ ]:
print("Per-class breakdown, competition-5 classes:")
display(per_class_df.sort_values(["model", "tier", "run", "class"]).reset_index(drop=True) if len(per_class_rows) else "Nothing completed yet.")

In [ ]:
print(f"Not yet complete ({len(missing)} of {len(EXPERIMENTS)} registered screening/headline slots):")
for m in missing:
    print(" -", m)

In [ ]:
import matplotlib.pyplot as plt

if len(summary_df):
    _agg = summary_df.groupby(["model", "tier"])["AUC"].mean().reset_index()
    _agg["label"] = _agg["model"] + " (" + _agg["tier"] + ")"
    _agg = _agg.sort_values("AUC")
    _colors = ["#0DA0BE" if t == "headline" else "#8D55AA" for t in _agg["tier"]]
    plt.figure(figsize=(8, max(3, 0.4 * len(_agg))))
    plt.barh(_agg["label"], _agg["AUC"], color=_colors)
    plt.xlabel("Mean AUC")
    plt.title("Completed experiments -- mean AUC (teal = headline, violet = screening)")
    plt.xlim(0, 1)
    plt.tight_layout()
    plt.show()
else:
    print("Nothing completed yet -- nothing to plot.")